# Giải bài tập MCQ

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


## Load dữ liệu

In [2]:
data_dir = '../Data/'
orders = pd.read_csv(data_dir + 'orders.csv', parse_dates=['order_date'], low_memory=False)
products = pd.read_csv(data_dir + 'products.csv', low_memory=False)
returns = pd.read_csv(data_dir + 'returns.csv', low_memory=False)
web_traffic = pd.read_csv(data_dir + 'web_traffic.csv', low_memory=False)
order_items = pd.read_csv(data_dir + 'order_items.csv', low_memory=False)
customers = pd.read_csv(data_dir + 'customers.csv', low_memory=False)
payments = pd.read_csv(data_dir + 'payments.csv', low_memory=False)
print('Đã tải xong toàn bộ dữ liệu.')

Đã tải xong toàn bộ dữ liệu.


### Q1: Trung vị số ngày giữa hai lần mua liên tiếp

In [3]:
# Sắp xếp theo khách hàng và ngày
orders_sorted = orders.sort_values(by=['customer_id', 'order_date'])

# Lọc khách hàng có nhiều hơn 1 đơn (Dùng duplicated để tối ưu hơn isin)
df_multi = orders_sorted[orders_sorted.duplicated(subset='customer_id', keep=False)].copy()

# Tính chênh lệch ngày giữa các đơn liên tiếp
df_multi['diff_days'] = df_multi.groupby('customer_id')['order_date'].diff().dt.days

# Tính trung vị
median_gap = df_multi['diff_days'].median()
print(f"Q1 Answer: {median_gap} ngày")

Q1 Answer: 144.0 ngày


### Q2: Tỷ suất lợi nhuận gộp trung bình theo segment

In [4]:
products['margin'] = (products['price'] - products['cogs']) / products['price']
margin_by_segment = products.groupby('segment')['margin'].mean().sort_values(ascending=False)
print("Q2 Answer:")
print(margin_by_segment)

Q2 Answer:
segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: margin, dtype: float64


### Q3: Lý do trả hàng nhiều nhất cho danh mục Streetwear

In [5]:
ret_prod = returns.merge(products, on='product_id')
streetwear_returns = ret_prod[ret_prod['category'] == 'Streetwear']
top_reason = streetwear_returns['return_reason'].value_counts()
print("Q3 Answer:")
print(top_reason)

Q3 Answer:
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64


### Q4: Nguồn traffic có tỷ lệ thoát trung bình thấp nhất

In [6]:
bounce_rate = web_traffic.groupby('traffic_source')['bounce_rate'].mean().sort_values()
print("Q4 Answer:")
print(bounce_rate)

Q4 Answer:
traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64


### Q5: Tỷ lệ order_items có khuyến mãi

In [7]:
has_promo = order_items['promo_id'].notnull().sum()
total_items = len(order_items)
promo_ratio = has_promo / total_items * 100
print(f"Q5 Answer: {promo_ratio:.2f}%")

Q5 Answer: 38.66%


### Q6: Nhóm tuổi có số đơn hàng trung bình cao nhất

In [8]:
print('Đang xử lý Q6...')
orders_per_cust = orders['customer_id'].value_counts().reset_index()
orders_per_cust.columns = ['customer_id', 'order_count']
cust_orders = customers.merge(orders_per_cust, on='customer_id', how='left')
cust_orders['order_count'] = cust_orders['order_count'].fillna(0)
age_group_orders = cust_orders[cust_orders['age_group'].notnull()].groupby('age_group')['order_count'].mean().sort_values(ascending=False)
print("Q6 Answer:")
print(age_group_orders)

Đang xử lý Q6...
Q6 Answer:
age_group
55+      5.406851
45-54    5.357241
35-44    5.337343
25-34    5.245226
18-24    5.226656
Name: order_count, dtype: float64


### Q7: Vùng có doanh thu cao nhất

In [9]:
print('Đang xử lý Q7...')
try:
    sales_train = pd.read_csv(data_dir + 'sales.csv', low_memory=False)
    geography = pd.read_csv(data_dir + 'geography.csv', low_memory=False)
    sales_geo = sales_train.merge(geography, on='location_id', how='inner')
    region_revenue = sales_geo.groupby('region')['revenue'].sum().sort_values(ascending=False)
    print("Q7 Answer:")
    print(region_revenue)
except Exception as e:
    print("Q7 Error:", e)

Đang xử lý Q7...


Q7 Error: 'location_id'


### Q8: Phương thức thanh toán nhiều nhất trong đơn huỷ

In [ ]:
cancelled_orders = orders[orders['order_status'] == 'cancelled']
cancelled_payments = cancelled_orders.merge(payments, on='order_id')
top_payment = cancelled_payments['payment_method'].value_counts()
print("Q8 Answer:")
print(top_payment)

Đang xử lý Q8...


KeyError: 'payment_method'

### Q9: Kích thước sản phẩm có tỷ lệ trả hàng cao nhất

In [ ]:
print('Đang xử lý Q9...')
items_with_size = order_items.merge(products[['product_id', 'size']], on='product_id')
size_counts = items_with_size['size'].value_counts()
returns_with_size = returns.merge(products[['product_id', 'size']], on='product_id')
return_counts = returns_with_size['size'].value_counts()
return_ratio = (return_counts / size_counts).dropna().sort_values(ascending=False)
print("Q9 Answer:")
print(return_ratio)

### Q10: Kế hoạch trả góp có giá trị thanh toán trung bình cao nhất

In [12]:
avg_payment_installments = payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)
print("Q10 Answer:")
print(avg_payment_installments)
print('--- HOÀN THÀNH ---')

Q10 Answer:
installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64
--- HOÀN THÀNH ---
